# Laboratorio de Modelos de Clasificación — Ejercicio 3: Breast Cancer

**Plantilla de implementación grupal | Google Colab | Python 3**

Este notebook está **intencionalmente sin código implementado** en las secciones de trabajo. Cada integrante debe completar únicamente las celdas de código que le fueron asignadas y respetar los nombres de variables acordados. Las celdas de texto explican el trabajo requerido, sus dependencias y cómo comprobar que quedó listo.

> **Alcance acordado para el equipo (no es una instrucción adicional atribuida al profesor):** desarrollar el ejercicio original de clasificación de cáncer de mama de principio a fin, con exploración de datos, preprocesamiento, cinco clasificadores individuales y una comparación homogénea. No se incorporan por defecto los cuatro escaladores ni los cinco híbridos del Ejercicio 1 (Santander) porque **no se proporcionó esa indicación para este ejercicio**.

## Organización del trabajo

| Integrante | Secciones / subsecciones asignadas | Entrega principal |
|---|---|---|
| **Marco** *(coordinación)* | 0.1–0.2, 5.1, 5.5, 6.3 y 7.1 | Entorno, contratos, evaluación central, análisis de errores e integración final |
| **Poma** | 1.1–1.3, 5.2 | Carga, validación de datos, variables predictoras y tabla comparativa |
| **Aldana** | 2.1–2.3, 4.1, 5.4, 6.2 | EDA, regresión logística, curvas ROC e interpretación general |
| **Brickzon** | 3.1–3.3, 4.2–4.3 | División estratificada, escalamiento, KNN y SVM |
| **Vera** | 4.4–4.5, 5.3 y 6.1 | Árboles, matrices de confusión e importancia de variables |

**Orden de integración:** Marco (0) → Poma (1) → Aldana (2) → Brickzon (3) → modeladores (4) → responsables de evaluación (5) → cierre (6–7).

Se puede desarrollar código por separado, pero la versión integrada se debe ejecutar **de arriba abajo**, sin depender de variables ocultas de una sesión anterior.

## Descripción del conjunto de datos

El notebook recibido describe el conjunto **Breast Cancer Wisconsin (Diagnostic)**. Las características se calculan sobre imágenes digitalizadas de aspirados con aguja fina (FNA) de masas mamarias y resumen propiedades geométricas y de textura de los núcleos celulares.

- **Problema:** clasificación supervisada binaria, `M` = maligno y `B` = benigno.
- **Observaciones descritas:** 569 (212 malignas y 357 benignas).
- **Predictores descritos:** 30 variables numéricas obtenidas a partir de diez mediciones: radio, textura, perímetro, área, suavidad, compacidad, concavidad, puntos cóncavos, simetría y dimensión fractal; cada medición aparece en forma media, error estándar y valor extremo (worst).
- **Datos de origen:** incluye identificación y diagnóstico. El identificador **no se usa como predictor**. Si aparece una columna vacía de exportación como `Unnamed: 32`, debe descartarse.
- **Valores ausentes:** la descripción original indica que no hay, pero deben comprobarlo en el archivo real.
- **Referencia del notebook original:** [UCI Breast Cancer Wisconsin (Diagnostic)](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic).

**Interpretación de clases acordada para este trabajo:** `1 = M (maligno, clase positiva)` y `0 = B (benigno)`. Es obligatorio respetar esta codificación en entrenamiento, métricas, curvas y matrices de confusión. **Precaución:** en `sklearn.datasets.load_breast_cancer()` la codificación de origen es inversa (`0 = malignant`, `1 = benign`); al usar esa alternativa, hay que convertirla.

Este laboratorio es académico: las predicciones de los modelos **no** son diagnósticos clínicos.

## Cómo usar esta plantilla en Google Colab

1. Abran este `.ipynb` en Google Colab y seleccionen el entorno de ejecución **Python 3 (CPU)**; no se requiere GPU.
2. **Opción preferida:** carguen el CSV original de la práctica en la ruta `/content/dataset/data.csv` (equivalente a `dataset/data.csv` si se ejecuta desde `/content`). Pueden crearse la carpeta `dataset` y subir el archivo desde el panel lateral de Colab. Si lo mantienen en Drive, deberán montar Drive y adaptar la ruta en **1.1**.
3. **Alternativa cuando no se dispone del CSV:** en **1.1**, usar `sklearn.datasets.load_breast_cancer(as_frame=True)` y construir un `df_raw` equivalente. **No mezclar fuentes de datos en una misma ejecución.** Documentar cuál se utilizó.
4. Completen las celdas de código vacías que tienen asignadas. No cambien los nombres compartidos definidos en **0.2** sin acordarlo con el equipo.
5. Cuando todas las partes estén listas: **Entorno de ejecución → Reiniciar sesión y ejecutar todo**. Cualquier error en la ejecución desde cero debe corregirse antes de entregar.

**Importante para el trabajo colaborativo:** Colab no fusiona automáticamente dos copias descargadas de un notebook. Coordinen una versión principal y hagan integración secuencial, o utilicen control de versiones para evitar sobrescribir trabajo ajeno.

## 0. Preparación del entorno y acuerdos de integración

### 0.1. Librerías y configuración para Google Colab

**Responsable: Marco.**

**Qué debes hacer**

- Importa las librerías comunes: `pandas`, `numpy`, `matplotlib.pyplot`, `seaborn` y herramientas necesarias de `scikit-learn` para separación, transformación, modelos y métricas.
- Comprueba que el entorno usa Python 3 y que las dependencias están disponibles en Colab. Instala únicamente dependencias ausentes; evita instalar paquetes sin necesidad.
- Fija `RANDOM_STATE = 42` y una semilla global de NumPy para favorecer reproducibilidad. Define preferencias visuales legibles para figuras.

**Resultado esperado / criterio de aceptación**

Una ejecución limpia de esta celda deja listos los imports y `RANDOM_STATE`. No depende de ninguna celda posterior.

In [ ]:
# 0.1 Librerías y configuración para Google Colab
# Responsable: Marco
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Herramientas scikit-learn
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    roc_curve,
)

# Verificación de entorno Python 3
assert sys.version_info >= (3, 8), f"Se requiere Python 3.8+; versión detectada: {sys.version}"

# Semilla fija para máxima reproducibilidad
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Preferencias visuales para figuras
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["font.size"] = 10

print(f"Librerías cargadas exitosamente bajo Python {sys.version.split()[0]}.")
print(f"Semilla de reproducibilidad global fijada: RANDOM_STATE = {RANDOM_STATE}")


### 0.2. Variables compartidas y registro de modelos

**Responsable: Marco.**

**Qué debes hacer**

- Declara los contenedores comunes `modelos_entrenados = {}` y `tipo_entrada = {}` antes del entrenamiento.
- Fija y comunica este contrato: `df_raw` = carga original; `df_clean` = columnas limpias; `X` = predictores numéricos; `y` = objetivo codificado con M=1/B=0; `X_train`, `X_test`, `y_train`, `y_test` = particiones; `escalador`, `X_train_scaled`, `X_test_scaled` = transformación ajustada solo en train.
- Cada responsable de un clasificador registrará `modelos_entrenados["Nombre del modelo"] = modelo_ajustado` y `tipo_entrada["Nombre del modelo"] = "escalado"` o `"original"`.
- Los nombres de modelo se acuerdan exactamente así: `Regresión logística`, `KNN`, `SVM`, `Árbol de decisión`, `Random Forest`.

**Resultado esperado / criterio de aceptación**

Existen dos diccionarios vacíos y el equipo conoce nombres y tipos. **No** crear aquí datos artificiales ni entrenar modelos.

In [ ]:
# 0.2 Variables compartidas y registro de modelos
# Responsable: Marco

# Contenedores compartidos para los modelos ajustados y la modalidad de entrada
modelos_entrenados = {}
tipo_entrada = {}

# Contrato de nombres de modelos acordado para el ejercicio
NOMBRES_MODELOS = [
    "Regresión logística",
    "KNN",
    "SVM",
    "Árbol de decisión",
    "Random Forest"
]

print("Contenedores comunes inicializados:")
print(" - modelos_entrenados:", modelos_entrenados)
print(" - tipo_entrada:", tipo_entrada)
print("\nContrato de integración:")
print(" - Nombres exactos de modelos:", NOMBRES_MODELOS)
print(" - Tipo de entrada admitido: 'escalado' (Logística, KNN, SVM) u 'original' (Árbol, Random Forest)")
print(" - Codificación objetivo: 1 = M (maligno, clase positiva de interés), 0 = B (benigno)")


## 1. Conjunto de datos

### 1.1. Carga del conjunto de datos

**Responsable: Poma.**

**Qué debes hacer**

- Comprueba si existe `/content/dataset/data.csv` y carga el CSV original del ejercicio en un DataFrame `df_raw`.
- Si el CSV no está disponible, utiliza el dataset Breast Cancer Wisconsin (Diagnostic) integrado en scikit-learn (`load_breast_cancer`) y genera un `df_raw` con predictores y una columna `diagnosis` que contenga etiquetas `M` y `B`.
- Muestra las primeras cinco filas, el número de registros y columnas y una nota visible que identifique la fuente utilizada (`CSV` o `sklearn`).

**Resultado esperado / criterio de aceptación**

Al terminar, `df_raw` existe, contiene una columna de diagnóstico y representa **una sola fuente** de datos.

**Atención:** No se debe obtener el diagnóstico de una variable predictora. Si se usa sklearn, recuerda que el target original tiene codificación inversa.

In [ ]:
# 1.1 Carga del CSV original (o alternativa reproducible de sklearn)
from pathlib import Path
import pandas as pd
from IPython.display import display

# En Colab se prefiere /content/dataset/data.csv; el segundo path permite
# ejecutar desde la carpeta Semana 13 del repositorio.
rutas_csv = [
    Path("/content/dataset/data.csv"),
    Path("dataset/data.csv"),
    Path("Semana 13/dataset/data.csv"),
]
ruta_csv = next((ruta for ruta in rutas_csv if ruta.is_file()), None)

if ruta_csv is not None:
    df_raw = pd.read_csv(ruta_csv)
    fuente_datos = f"CSV: {ruta_csv}"
else:
    from sklearn.datasets import load_breast_cancer
    datos_bc = load_breast_cancer(as_frame=True)
    df_raw = datos_bc.data.copy()
    # sklearn codifica 0=maligno, 1=benigno; convertir a etiquetas explícitas.
    df_raw["diagnosis"] = datos_bc.target.map({0: "M", 1: "B"})
    fuente_datos = "sklearn.datasets.load_breast_cancer (alternativa; 0 original=M, 1 original=B)"

if "diagnosis" not in df_raw.columns:
    raise ValueError("El conjunto debe contener una columna 'diagnosis'.")

print(f"Fuente utilizada: {fuente_datos}")
print(f"Dimensiones originales: {df_raw.shape[0]} filas × {df_raw.shape[1]} columnas")
display(df_raw.head())


### 1.2. Validación inicial y limpieza

**Responsable: Poma.**

**Qué debes hacer**

- Revisa tipos de datos, duplicados, valores nulos y distribución de las clases.
- Elimina columnas que no deben ser predictoras (`id`, `Unnamed: 32` u otras columnas enteramente vacías, si aparecen). Conserva `diagnosis` para formar el objetivo.
- Si existen ausentes, explica el criterio aplicado. Si se requiere imputación estadística, **no** calcules parámetros con el conjunto completo: esa imputación se debe ajustar usando únicamente entrenamiento en la sección 3.
- Guarda el resultado tabular en `df_clean`, sin modificar permanentemente `df_raw`.

**Resultado esperado / criterio de aceptación**

`df_clean` contiene `diagnosis` más las variables predictoras válidas. Reporta el antes/después y las decisiones de limpieza.

**Atención:** En caso de duplicados, justificar su tratamiento; no eliminar muestras automáticamente sin documentar por qué.

In [ ]:
# 1.2 Validación inicial y limpieza sin alterar df_raw
import pandas as pd
from IPython.display import display

print("Dimensiones antes de limpiar:", df_raw.shape)
print("Tipos de datos originales:")
display(df_raw.dtypes.rename("tipo").to_frame())
print("Filas duplicadas exactas:", int(df_raw.duplicated().sum()))
print("Valores ausentes por columna (solo columnas con ausentes):")
ausentes = df_raw.isna().sum()
display(ausentes[ausentes.gt(0)].rename("ausentes").to_frame())
print("Distribución original de diagnosis:")
display(df_raw["diagnosis"].value_counts(dropna=False).rename("conteo").to_frame())

# Se trabaja sobre una copia. Los identificadores y columnas de exportación vacías
# no describen características y no deben entrar a X.
df_clean = df_raw.copy()
df_clean.columns = [str(col).strip() for col in df_clean.columns]
columnas_no_predictoras = [
    col for col in df_clean.columns
    if col.lower() == "id"
    or col.lower().startswith("unnamed:")
    or (col.lower() != "diagnosis" and df_clean[col].isna().all())
]
df_clean = df_clean.drop(columns=columnas_no_predictoras)

if "diagnosis" not in df_clean.columns:
    raise ValueError("No se encontró la columna diagnosis después de la limpieza.")
df_clean["diagnosis"] = df_clean["diagnosis"].astype("string").str.strip().str.upper()
if df_clean["diagnosis"].isna().any():
    raise ValueError("Hay diagnósticos ausentes; no se eliminan filas de forma silenciosa.")

etiquetas_observadas = set(df_clean["diagnosis"].unique())
if not etiquetas_observadas.issubset({"M", "B"}):
    raise ValueError(f"Etiquetas de diagnóstico inesperadas: {sorted(etiquetas_observadas)}")

print("Columnas eliminadas (ID/exportación):", columnas_no_predictoras or "ninguna")
print("Filas duplicadas conservadas para no descartar muestras sin justificación:", int(df_clean.duplicated().sum()))
print("Dimensiones después de limpiar:", df_clean.shape)
print("Valores ausentes restantes:", int(df_clean.isna().sum().sum()))
print("Criterio de ausentes: no imputar antes de la partición; si hubiera ausentes en predictores, la imputación debe ajustarse solo con train en 3.2.")


### 1.3. Selección de X e y y codificación del diagnóstico

**Responsable: Poma.**

**Qué debes hacer**

- Crea `y` desde `df_clean["diagnosis"]` usando la convención `M → 1` y `B → 0`. Si el objetivo ya está numérico, valida explícitamente su significado antes de convertirlo.
- Crea `X` con **solo** las 30 características predictoras numéricas; excluye diagnóstico, identificadores y campos derivados de la etiqueta.
- Valida que `X` y `y` tienen el mismo número de filas, que `X` no incluye el diagnóstico, que `y` contiene únicamente `{0, 1}` y que no hay infinitos ni textos en `X`.
- Muestra forma de `X`, conteo de `y` y lista de columnas predictoras.

**Resultado esperado / criterio de aceptación**

Se pueden usar `X` e `y` en todas las siguientes secciones. Debe quedar claro que **1 = maligno**.

In [ ]:
# 1.3 Selección de predictores y codificación M=1 / B=0
import numpy as np
import pandas as pd

if "diagnosis" not in df_clean.columns:
    raise ValueError("df_clean debe contener diagnosis.")

# La etiqueta se mapea explícitamente; nunca se infiere desde una característica.
y = df_clean["diagnosis"].map({"M": 1, "B": 0}).astype("int64")
X = df_clean.drop(columns=["diagnosis"]).copy()

# El contrato del ejercicio requiere las 30 características numéricas originales.
columnas_no_numericas = X.select_dtypes(exclude=[np.number]).columns.tolist()
if columnas_no_numericas:
    raise TypeError(f"Predictores no numéricos detectados: {columnas_no_numericas}")
if X.shape[1] != 30:
    raise ValueError(f"Se esperaban 30 predictores; se encontraron {X.shape[1]}: {X.columns.tolist()}")

valores_X = X.to_numpy(dtype=float)
if len(X) != len(y):
    raise ValueError("X e y no tienen el mismo número de filas.")
if "diagnosis" in X.columns:
    raise AssertionError("La etiqueta no debe formar parte de X.")
if not set(y.unique()).issubset({0, 1}):
    raise ValueError("y debe contener únicamente 0 y 1.")
if np.isinf(valores_X).any():
    raise ValueError("X contiene valores infinitos; deben corregirse antes de entrenar.")

conteo_ausentes = X.isna().sum()
columnas_con_ausentes = conteo_ausentes[conteo_ausentes.gt(0)]
print("Codificación objetivo: 1 = M (maligno), 0 = B (benigno)")
print("Forma de X:", X.shape)
print(f"Valores ausentes en X: {int(conteo_ausentes.sum())}")
if not columnas_con_ausentes.empty:
    print("Ausentes por predictor:")
    display(columnas_con_ausentes.rename("ausentes").to_frame())
print("Los ausentes se reservan para imputación en 3.2, ajustada únicamente con X_train.")
print("Conteo de y (0=benigno, 1=maligno):")
display(y.value_counts().sort_index().rename(index={0: "0 = B", 1: "1 = M"}).rename("conteo").to_frame())
print("Predictores (30):")
print(X.columns.tolist())


## 2. Análisis exploratorio de datos (EDA)

### 2.1. Distribución de clases y balance del objetivo

**Responsable: Aldana.**

**Qué debes hacer**

- Usa `y` para calcular frecuencias absolutas y relativas de benignos (0) y malignos (1).
- Presenta un gráfico de barras con etiquetas legibles y porcentajes.
- Escribe una conclusión breve sobre la diferencia de proporciones y por qué puede influir en Accuracy y Recall.

**Resultado esperado / criterio de aceptación**

Tabla o impresión de conteos + figura + interpretación de 2–4 líneas. No modificar `X` ni `y`.

### 2.2. Estadísticas descriptivas y distribución de predictores

**Responsable: Aldana.**

**Qué debes hacer**

- Muestra estadísticas descriptivas de `X` e inspecciona rangos y posibles valores extremos.
- Genera histogramas y/o boxplots de **un conjunto razonado** de variables (por ejemplo, radio, área, textura y concavidad), sin saturar el notebook con 30 gráficos redundantes.
- Compara visualmente al menos dos variables entre diagnósticos benigno y maligno, manteniendo la convención de colores/etiquetas en cada gráfico.
- Registra hallazgos observables, sin afirmar causalidad.

**Resultado esperado / criterio de aceptación**

Estadísticas descriptivas, gráficos legibles y conclusiones breves sobre variabilidad y diferencias visuales.

### 2.3. Correlación y posibles redundancias

**Responsable: Aldana.**

**Qué debes hacer**

- Calcula la matriz de correlación de las 30 características de `X`.
- Muestra un mapa de calor legible (completo o filtrado a correlaciones altas) e identifica algunos pares de variables correlacionadas.
- Explica cómo la multicolinealidad puede afectar la interpretación de algunos clasificadores, y aclara que correlación **no** implica causalidad.
- No elimines columnas todavía sin acordarlo con Poma y Brickzon.

**Resultado esperado / criterio de aceptación**

Visualización de correlación y explicación fundamentada de 2–4 hallazgos.

**Atención:** El EDA es exploratorio. Cualquier selección automática de variables dependiente de datos deberá ajustarse en entrenamiento, nunca usando test.

## 3. Modificaciones y preprocesamiento

### 3.1. División estratificada entrenamiento/prueba

**Responsable: Brickzon.**

**Qué debes hacer**

- Utiliza `train_test_split` con `test_size=0.20`, `random_state=RANDOM_STATE` y `stratify=y`.
- Crea **exactamente** `X_train`, `X_test`, `y_train`, `y_test`.
- Verifica tamaños y porcentajes de M/B en ambos subconjuntos. No alteres el test después de la división.

**Resultado esperado / criterio de aceptación**

Partición reproducible y estratificada; el conjunto de prueba se reserva para la evaluación final.

**Atención:** No escalar, imputar mediante estadísticas ni remuestrear antes de separar: eso causaría fuga de datos.

In [ ]:
# 3.1 División estratificada entrenamiento/prueba
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from IPython.display import display

if "RANDOM_STATE" not in globals():
    RANDOM_STATE = 42

if "X" not in globals() or "y" not in globals():
    raise NameError("Las variables 'X' e 'y' deben estar definidas antes de particionar (ejecuta la sección 1.3).")

# Partición estratificada 80% train / 20% test con semilla global
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

# Validaciones defensivas de consistencia
if len(X_train) != len(y_train):
    raise ValueError("X_train e y_train no tienen la misma cantidad de filas.")
if len(X_test) != len(y_test):
    raise ValueError("X_test e y_test no tienen la misma cantidad de filas.")
if X_train.shape[1] != 30 or X_test.shape[1] != 30:
    raise ValueError(f"Se esperaban 30 predictores; train={X_train.shape[1]}, test={X_test.shape[1]}")

# Resumen detallado de estratificación
resumen_particion = pd.DataFrame({
    "Total muestras": [len(y), len(y_train), len(y_test)],
    "Porcentaje split (%)": [100.0, len(y_train) / len(y) * 100, len(y_test) / len(y) * 100],
    "Benigno (0)": [int((y == 0).sum()), int((y_train == 0).sum()), int((y_test == 0).sum())],
    "Maligno (1)": [int((y == 1).sum()), int((y_train == 1).sum()), int((y_test == 1).sum())],
    "Prop. Maligno (%)": [
        (y == 1).mean() * 100,
        (y_train == 1).mean() * 100,
        (y_test == 1).mean() * 100
    ]
}, index=["Dataset completo", "Entrenamiento (train)", "Prueba (test)"])

print("Partición estratificada 80/20 completada con éxito:")
display(resumen_particion.round(2))
print("El conjunto de prueba (test) queda estrictamente reservado para la evaluación final en la sección 5.")


### 3.2. Escalamiento sin fuga de información

**Responsable: Brickzon.**

**Qué debes hacer**

- Elige `StandardScaler` como transformación inicial para regresión logística, KNN y SVM. Explica brevemente por qué estos modelos pueden ser sensibles a escalas distintas.
- Ajusta `escalador` con `fit` **solo** sobre `X_train`, y transforma ambos subconjuntos en `X_train_scaled` y `X_test_scaled` sin cambiar el orden ni la cantidad de filas.
- Conserva `X_train` y `X_test` sin escalar para Árbol de decisión y Random Forest.
- Si se necesita imputación, incorpora correctamente la imputación ajustada solo en train; idealmente construye un `Pipeline` coherente y evita cambiar el contrato de variables sin avisar.

**Resultado esperado / criterio de aceptación**

Existen `escalador`, `X_train_scaled` y `X_test_scaled`, compatibles con los modelos y sin uso del test para ajustar parámetros.

**Atención:** No es necesario implementar cuatro escaladores en este Ejercicio 3: es una propuesta independiente del Ejercicio 1.

In [ ]:
# 3.2 Escalamiento e imputación condicional sin fuga de información
# Responsable: Brickzon (@Marcos)
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from IPython.display import display

if "X_train" not in globals() or "X_test" not in globals():
    raise NameError("Ejecuta primero la sección 3.1 para generar X_train y X_test.")

# 1. Estrategia condicional de imputación (ajustada estrictamente sobre X_train):
# Poma reservó los valores ausentes en 1.2/1.3 para resolverse aquí sin fuga de datos.
# Si aparecen ausentes en X_train o X_test, se imputan usando la mediana calculada solo en train,
# garantizando compatibilidad con modelos lineales/distancias y con modelos basados en árboles.
conteo_ausentes_train = int(X_train.isna().sum().sum())
conteo_ausentes_test = int(X_test.isna().sum().sum())
imputador = None

if conteo_ausentes_train > 0 or conteo_ausentes_test > 0:
    imputador = SimpleImputer(strategy="median")
    imputador.fit(X_train)
    X_train = pd.DataFrame(
        imputador.transform(X_train),
        columns=X_train.columns,
        index=X_train.index
    )
    X_test = pd.DataFrame(
        imputador.transform(X_test),
        columns=X_test.columns,
        index=X_test.index
    )
    print(f"Imputación condicional aplicada (SimpleImputer, mediana): {conteo_ausentes_train} ausentes en train, {conteo_ausentes_test} en test.")
else:
    print("Sin valores ausentes en X_train ni X_test. No fue necesario aplicar imputación activa.")

# 2. Justificación del escalamiento:
# - KNN se basa en distancia euclidiana: predictores con gran rango numérico (ej. area_mean ~ 2500) dominarían frente a rangos pequeños (ej. smoothness ~ 0.1).
# - SVM con kernel RBF o lineal maximiza márgenes geométricos; variables con escalas dispares distorsionan los hiperplanos.
# - Regresión Logística regularizada aplica penalizaciones L1/L2 por igual, perjudicando injustamente a predictores con menor escala.
# - Modelos de ensamble/árboles (Decision Tree y Random Forest) son invariantes a transformaciones monótonas y usan X_train y X_test originales (ya limpios/imputados).

escalador = StandardScaler()

# Ajuste estrictamente sobre X_train para evitar fuga de información desde test
escalador.fit(X_train)

# Transformación conservando DataFrame con nombres de columnas e índices correspondientes
X_train_scaled = pd.DataFrame(
    escalador.transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)
X_test_scaled = pd.DataFrame(
    escalador.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

# Validación de propiedades estadísticas en entrenamiento (media ~ 0, varianza ~ 1)
medias_train = X_train_scaled.mean()
varianzas_train = X_train_scaled.var()

print("Escalamiento con StandardScaler ejecutado sin fuga de datos:")
print(f" - Escalador ajustado exclusivamente con X_train ({X_train.shape[0]} observaciones).")
print(f" - Forma de X_train_scaled: {X_train_scaled.shape}")
print(f" - Forma de X_test_scaled:  {X_test_scaled.shape}")
print(f" - Promedio de medias en train escalado:     {medias_train.mean():.4f} (teórico: 0)")
print(f" - Promedio de varianzas en train escalado:  {varianzas_train.mean():.4f} (teórico: 1)")
print(" - Variables X_train y X_test (sin escalar, compatibles con árboles) preservadas.")


### 3.3. Comprobaciones de preparación antes de entrenar

**Responsable: Brickzon.**

**Qué debes hacer**

- Comprueba que `X_train_scaled` y `X_test_scaled` mantienen los mismos números de filas y columnas que sus equivalentes originales.
- Verifica ausencia de NaN e infinitos en los arreglos que se usarán para entrenar y predecir.
- Comprueba que las etiquetas de ambas particiones respetan `1 = M`.
- Incluye una breve salida de verificación para indicar que ya se puede empezar la sección 4.

**Resultado esperado / criterio de aceptación**

Sin errores de forma, valores inválidos ni codificación de clases; todas las variables listas para los cinco clasificadores.

In [ ]:
# 3.3 Comprobaciones de preparación antes de entrenar
# Responsable: Brickzon (@Marcos)
import numpy as np

# 1. Comprobación de dimensiones idénticas
if X_train_scaled.shape != X_train.shape:
    raise AssertionError(f"Dimensión inconsistente en train: {X_train_scaled.shape} vs {X_train.shape}")
if X_test_scaled.shape != X_test.shape:
    raise AssertionError(f"Dimensión inconsistente en test: {X_test_scaled.shape} vs {X_test.shape}")

# 2. Ausencia de NaN e infinitos en arreglos de entrenamiento y prueba (originales y escalados)
# Confirma que la estrategia de imputación condicional de 3.2 garantizó matrices libres de ausentes
for nombre_arr, arr in [
    ("X_train", X_train.to_numpy(dtype=float)),
    ("X_test", X_test.to_numpy(dtype=float)),
    ("X_train_scaled", X_train_scaled.to_numpy(dtype=float)),
    ("X_test_scaled", X_test_scaled.to_numpy(dtype=float)),
]:
    if not np.isfinite(arr).all():
        raise ValueError(f"{nombre_arr} contiene valores NaN o infinitos tras el preprocesamiento.")

# 3. Validación de etiquetas binarias respetando 1 = M (maligno) y 0 = B (benigno)
if not set(np.unique(y_train)).issubset({0, 1}) or not set(np.unique(y_test)).issubset({0, 1}):
    raise ValueError("Las etiquetas y_train e y_test deben contener únicamente los valores {0, 1}.")

# 4. Estado del flujo de imputación
estado_imputacion = "Aplicada en 3.2" if "imputador" in globals() and imputador is not None else "No requerida (0 ausentes)"

print("============================================================")
print("✓ VERIFICACIONES DE PREPROCESAMIENTO COMPLETADAS CON ÉXITO")
print("============================================================")
print(f"- Dimensiones de entrenamiento: X_train {X_train.shape}, y_train {y_train.shape}")
print(f"- Dimensiones de prueba:        X_test {X_test.shape}, y_test {y_test.shape}")
print(f"- Flujo de imputación:          {estado_imputacion}")
print("- Integridad numérica:          0 valores NaN y 0 infinitos en datos originales y escalados.")
print(f"- Codificación de clases:       1 = M (maligno: {(y_train == 1).sum()} train / {(y_test == 1).sum()} test), 0 = B (benigno)")
print("- Datos listos para modelos lineales/distancias (escalados) y modelos de árboles (sin escalar).")


## 4. Modelos de clasificación

**Reglas comunes para los responsables de 4.1–4.5**

- Entrenar **solo** con la partición de entrenamiento. Usar `RANDOM_STATE` cuando el estimador lo permita.
- Mantener la clase **M = 1** como positiva.
- Registrar el modelo ya ajustado en `modelos_entrenados` con **el nombre exacto** establecido en 0.2, y el tipo de entrada en `tipo_entrada` (`"escalado"` u `"original"`).
- No reescribir `X_train`, `X_test`, `y_train`, `y_test` ni cambiar el escalador de otros integrantes.
- Mostrar una evidencia pequeña de entrenamiento correcto (por ejemplo, nombre del modelo, hiperparámetros principales o predicciones de ejemplo). **La evaluación comparativa del conjunto de prueba se centraliza en la sección 5**, no en cada modelo.
- Si se desean ajustar hiperparámetros, hacerlo solo con entrenamiento (validación cruzada/`GridSearchCV`), sin seleccionar modelos con el conjunto de prueba.

### 4.1. Regresión logística

**Responsable: Aldana.**

**Qué debes hacer**

- Instancia y entrena `LogisticRegression` usando `X_train_scaled` y `y_train`; configura un `max_iter` suficiente para evitar advertencias de convergencia.
- Registra el estimador bajo `modelos_entrenados["Regresión logística"]` y declara `tipo_entrada["Regresión logística"] = "escalado"`.
- Anota 1–2 líneas explicando por qué la regresión logística sirve como línea base interpretable.

**Resultado esperado / criterio de aceptación**

Regresión logística ajustada y registrada, lista para evaluar en la sección 5.

### 4.2. K vecinos más cercanos (KNN)

**Responsable: Brickzon.**

**Qué debes hacer**

- Entrena un `KNeighborsClassifier` con los datos escalados; indica el número de vecinos `n_neighbors` elegido.
- Si comparas varios valores de K, selecciónalos con validación **solo sobre entrenamiento**.
- Registra `modelos_entrenados["KNN"]` y `tipo_entrada["KNN"] = "escalado"`.
- Explica en 1–2 líneas el efecto de la distancia y por qué debe escalarse la entrada.

**Resultado esperado / criterio de aceptación**

KNN entrenado y registrado, sin utilizar el conjunto de prueba para elegir K.

In [ ]:
# 4.2 K vecinos más cercanos (KNN)
# Responsable: Brickzon (@Marcos)
import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

if "modelos_entrenados" not in globals():
    modelos_entrenados = {}
if "tipo_entrada" not in globals():
    tipo_entrada = {}

# Justificación del escalamiento:
# KNN asigna etiquetas según la distancia en el espacio de características.
# Con datos escalados, cada una de las 30 propiedades celulares contribuye proporcionalmente a la métrica de distancia.

# Selección de hiperparámetro k mediante validación cruzada (5-Fold CV)
# Para evitar fuga de información entre pliegues, se utiliza un Pipeline con StandardScaler
# y se evalúa pasando X_train SIN escalar a cross_val_score()
k_candidatos = [3, 5, 7, 9, 11]
scores_cv = {}

for k in k_candidatos:
    pasos_pipe = []
    if "imputador" in globals() and imputador is not None:
        from sklearn.impute import SimpleImputer
        pasos_pipe.append(("imputer", SimpleImputer(strategy="median")))
    pasos_pipe.append(("scaler", StandardScaler()))
    pasos_pipe.append(("knn", KNeighborsClassifier(n_neighbors=k, weights="uniform", metric="minkowski", p=2)))
    
    pipe_knn = Pipeline(pasos_pipe)
    
    # Se evalúa sobre X_train sin escalar; el Pipeline escala internamente en cada pliegue
    score_f1 = cross_val_score(pipe_knn, X_train, y_train, cv=5, scoring="f1").mean()
    scores_cv[k] = score_f1

k_optimo = max(scores_cv, key=scores_cv.get)
print("Selección de hiperparámetro K mediante validación cruzada (Pipeline + StandardScaler sobre X_train):")
for k, score in scores_cv.items():
    marca = " <-- k seleccionado (mejor F1 en CV)" if k == k_optimo else ""
    print(f" - k={k}: F1 medio en CV = {score:.4f}{marca}")

# Ajuste definitivo del modelo KNN final sobre X_train_scaled según contrato
modelo_knn = KNeighborsClassifier(n_neighbors=k_optimo, weights="uniform", metric="minkowski", p=2)
modelo_knn.fit(X_train_scaled, y_train)

# Registro oficial en los contenedores compartidos (contrato de 0.2)
modelos_entrenados["KNN"] = modelo_knn
tipo_entrada["KNN"] = "escalado"

print(f"\nModelo KNN final ajustado exitosamente con n_neighbors={k_optimo} sobre X_train_scaled.")
print(f"Registrado en modelos_entrenados['KNN'] con tipo_entrada = '{tipo_entrada['KNN']}'.")
print("Nota: La evaluación comparativa final sobre el conjunto de prueba se centraliza en la sección 5.")


### 4.3. Máquinas de vectores de soporte (SVM)

**Responsable: Brickzon.**

**Qué debes hacer**

- Entrena un `SVC` con los datos escalados; documenta el kernel y al menos un hiperparámetro (por ejemplo, `C`).
- Para curva ROC, asegúrate de poder recuperar una puntuación de la **clase 1** mediante `decision_function` o, si corresponde, `predict_proba` (`probability=True`).
- Registra `modelos_entrenados["SVM"]` y `tipo_entrada["SVM"] = "escalado"`.

**Resultado esperado / criterio de aceptación**

SVM entrenado, puntuación para ROC identificada y registro consistente con el resto.

In [ ]:
# 4.3 Máquinas de vectores de soporte (SVM)
# Responsable: Brickzon (@Marcos)
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

if "RANDOM_STATE" not in globals():
    RANDOM_STATE = 42
if "modelos_entrenados" not in globals():
    modelos_entrenados = {}
if "tipo_entrada" not in globals():
    tipo_entrada = {}

# Configuración del estimador SVC:
# - kernel='rbf': proyecta a un espacio de Hilbert para capturar límites de decisión no lineales.
# - C=1.0: equilibrio estándar entre margen amplio y penalización de errores de clasificación.
# - probability=True: habilita predict_proba para curvas ROC y calibración probabilística.
# - decision_function: disponible de forma nativa para puntuaciones continuas de discriminación de clase 1.
# - random_state=RANDOM_STATE: garantiza reproducibilidad en el ajuste de probabilidades internas.

# Validación cruzada (5-Fold CV) sin fuga de datos usando Pipeline con StandardScaler sobre X_train sin escalar
pasos_pipe_svm = []
if "imputador" in globals() and imputador is not None:
    from sklearn.impute import SimpleImputer
    pasos_pipe_svm.append(("imputer", SimpleImputer(strategy="median")))
pasos_pipe_svm.append(("scaler", StandardScaler()))
pasos_pipe_svm.append(("svm", SVC(C=1.0, kernel="rbf", gamma="scale", probability=True, random_state=RANDOM_STATE)))

pipe_svm = Pipeline(pasos_pipe_svm)
f1_cv_svm = cross_val_score(pipe_svm, X_train, y_train, cv=5, scoring="f1").mean()

# Ajuste definitivo del modelo SVM final sobre X_train_scaled según contrato
modelo_svm = SVC(
    C=1.0,
    kernel="rbf",
    gamma="scale",
    probability=True,
    random_state=RANDOM_STATE
)
modelo_svm.fit(X_train_scaled, y_train)

# Registro oficial en los contenedores compartidos (contrato de 0.2)
modelos_entrenados["SVM"] = modelo_svm
tipo_entrada["SVM"] = "escalado"

print("Modelo SVM ajustado exitosamente:")
print(f" - Hiperparámetros: kernel='{modelo_svm.kernel}', C={modelo_svm.C}, gamma='{modelo_svm.gamma}'")
print(f" - Vectores de soporte totales: {len(modelo_svm.support_)} ({modelo_svm.n_support_} por clase [B, M])")
print(f" - F1 medio en validación cruzada interna (Pipeline sobre X_train sin escalar): {f1_cv_svm:.4f}")
print(" - Puntuación continua disponible para Curva ROC (5.4): decision_function y predict_proba(X)[:, 1]")
print(f"Registrado en modelos_entrenados['SVM'] con tipo_entrada = '{tipo_entrada['SVM']}'.")
print("Nota: La evaluación comparativa final sobre el conjunto de prueba se centraliza en la sección 5.")


### 4.4. Árbol de decisión

**Responsable: Vera.**

**Qué debes hacer**

- Entrena `DecisionTreeClassifier` con `X_train` **sin escalar** y `y_train`.
- Controla al menos un parámetro de complejidad (por ejemplo, `max_depth` o `min_samples_leaf`) y justifica brevemente el criterio para limitar sobreajuste.
- Registra `modelos_entrenados["Árbol de decisión"]` y `tipo_entrada["Árbol de decisión"] = "original"`.

**Resultado esperado / criterio de aceptación**

Árbol ajustado, registrado y con control explícito de complejidad.

### 4.5. Random Forest

**Responsable: Vera.**

**Qué debes hacer**

- Entrena `RandomForestClassifier` con `X_train` sin escalar y `y_train`.
- Documenta `n_estimators` y, de usarlo, algún parámetro de regularización o profundidad.
- Registra `modelos_entrenados["Random Forest"]` y `tipo_entrada["Random Forest"] = "original"`.
- Confirma que el estimador expone `feature_importances_` para la interpretación de 6.1.

**Resultado esperado / criterio de aceptación**

Random Forest entrenado y registrado; quedan disponibles sus importancias de características.

## 5. Evaluación de los modelos

### 5.1. Predicciones y cálculo centralizado de métricas

**Responsable: Marco.**

**Qué debes hacer**

- Comprueba que `modelos_entrenados` contiene exactamente los cinco nombres acordados y que `tipo_entrada` tiene una entrada para cada uno.
- Para cada modelo selecciona `X_test_scaled` si su tipo es `"escalado"`, o `X_test` si es `"original"`; genera su predicción y una puntuación para ROC-AUC referida a `M = 1`.
- Calcula por modelo `Accuracy`, `Precision_M`, `Recall_M`, `F1_M` y `ROC_AUC` (usa `pos_label=1` cuando sea relevante y considera `zero_division=0` si corresponde).
- Guarda `resultados_df` con columnas `Modelo`, `Accuracy`, `Precision_M`, `Recall_M`, `F1_M`, `ROC_AUC` y define `y_pred_por_modelo`, `y_score_por_modelo` como diccionarios de resultados por nombre.
- Para el ROC-AUC usa **puntuaciones o probabilidades** de la clase maligna, nunca las etiquetas 0/1 de `predict` como sustituto de una puntuación.

**Resultado esperado / criterio de aceptación**

Existen `resultados_df`, `y_pred_por_modelo`, `y_score_por_modelo`; se ha evaluado **solo una vez** el test reservado y las cinco filas son comparables.

**Atención:** Para `predict_proba`, verificar el orden con `modelo.classes_`: no suponer que una columna fija corresponde siempre a maligno.

### 5.2. Tabla comparativa y ordenamiento

**Responsable: Poma.**

**Qué debes hacer**

- Muestra `resultados_df` redondeado de manera uniforme y ordénalo según `Recall_M` y como criterio de desempate `F1_M` o `ROC_AUC`.
- Destaca qué modelo obtiene el mejor valor por métrica, sin llamar mejor a un modelo solamente por Accuracy.
- Anota de forma breve si hay diferencias entre el mejor modelo por Recall y el mejor por ROC-AUC.

**Resultado esperado / criterio de aceptación**

Tabla clara de cinco modelos, comparación defendible y lectura correcta de la clase positiva.

In [ ]:
# 5.2 Tabla comparativa ordenada por Recall_M y luego F1_M
import numpy as np
import pandas as pd
from IPython.display import display

columnas_metricas = ["Accuracy", "Precision_M", "Recall_M", "F1_M", "ROC_AUC"]
columnas_requeridas = ["Modelo", *columnas_metricas]
if "resultados_df" not in globals():
    raise NameError("Ejecuta primero la sección 5.1 para crear resultados_df.")
if not set(columnas_requeridas).issubset(resultados_df.columns):
    raise ValueError(f"resultados_df debe incluir estas columnas: {columnas_requeridas}")

modelos_esperados = {
    "Regresión logística", "KNN", "SVM", "Árbol de decisión", "Random Forest"
}
if resultados_df["Modelo"].isna().any() or not resultados_df["Modelo"].map(lambda nombre: isinstance(nombre, str)).all():
    raise ValueError("La columna Modelo debe contener nombres de modelo no nulos como texto.")
if resultados_df["Modelo"].duplicated().any():
    repetidos = resultados_df.loc[resultados_df["Modelo"].duplicated(keep=False), "Modelo"].tolist()
    raise ValueError(f"resultados_df contiene modelos duplicados: {repetidos}")
modelos_observados = set(resultados_df["Modelo"])
if len(resultados_df) != 5 or modelos_observados != modelos_esperados:
    faltantes = modelos_esperados - modelos_observados
    inesperados = modelos_observados - modelos_esperados
    raise ValueError(
        "resultados_df debe contener exactamente los cinco clasificadores acordados. "
        f"Faltantes: {sorted(faltantes)}; inesperados: {sorted(inesperados)}; "
        f"filas: {len(resultados_df)}."
    )

comparacion = resultados_df[columnas_requeridas].copy()
comparacion[columnas_metricas] = comparacion[columnas_metricas].apply(pd.to_numeric, errors="raise")
if not np.isfinite(comparacion[columnas_metricas].to_numpy(dtype=float)).all():
    raise ValueError("Las métricas deben ser numéricas y finitas; se encontraron NaN o infinitos.")
comparacion = comparacion.sort_values(
    ["Recall_M", "F1_M", "ROC_AUC"], ascending=False, kind="mergesort"
).reset_index(drop=True)

print("Comparación de modelos (1 = maligno). Orden: Recall_M, F1_M y ROC_AUC descendentes.")
tabla_mostrada = comparacion.copy()
ganadores_por_metrica = {}
for metrica in columnas_metricas:
    maximo = comparacion[metrica].max()
    es_maximo = np.isclose(comparacion[metrica].to_numpy(dtype=float), maximo)
    ganadores_por_metrica[metrica] = comparacion.loc[es_maximo, "Modelo"].tolist()
    tabla_mostrada[metrica] = [
        f"{valor:.4f}*" if np.isclose(valor, maximo) else f"{valor:.4f}"
        for valor in comparacion[metrica]
    ]
display(tabla_mostrada)
print("* Mejor valor observado para esa métrica; se marcan todos los empates (np.isclose).")

print("Modelos líderes por métrica:")
for metrica, ganadores in ganadores_por_metrica.items():
    maximo = comparacion[metrica].max()
    print(f"- {metrica}: {', '.join(ganadores)} ({maximo:.4f})")

ganadores_recall = ganadores_por_metrica["Recall_M"]
ganadores_auc = ganadores_por_metrica["ROC_AUC"]
if set(ganadores_recall) == set(ganadores_auc):
    print(
        "Recall_M y ROC_AUC tienen el mismo grupo de líderes: "
        f"{', '.join(ganadores_recall)}."
    )
else:
    coinciden = [modelo for modelo in ganadores_recall if modelo in set(ganadores_auc)]
    print(
        f"Líderes de Recall_M: {', '.join(ganadores_recall)}; "
        f"líderes de ROC_AUC: {', '.join(ganadores_auc)}."
    )
    if coinciden:
        print(f"Líderes compartidos entre ambas métricas: {', '.join(coinciden)}.")
    else:
        print("No hay modelos empatados como líderes en ambas métricas.")


### 5.3. Matrices de confusión y falsos negativos

**Responsable: Vera.**

**Qué debes hacer**

- Para cada modelo, usa `y_test` y `y_pred_por_modelo[nombre]` para elaborar una matriz de confusión con orden de etiquetas `[0, 1]`.
- Etiqueta los ejes como valores reales y predichos, con `0=B` y `1=M`, de forma consistente.
- Identifica el cuadrante de **falsos negativos de malignidad** (real M=1, predicho B=0) y compara su cantidad en los cinco modelos.
- Incluye una interpretación breve de por qué los falsos negativos son relevantes para este ejercicio.

**Resultado esperado / criterio de aceptación**

Cinco matrices de confusión legibles y comparación explícita de falsos negativos para la clase maligna.

### 5.4. Curvas ROC comparadas

**Responsable: Aldana.**

**Qué debes hacer**

- Utiliza las puntuaciones registradas en `y_score_por_modelo` y las etiquetas verdaderas `y_test` para construir una curva ROC por clasificador.
- Incluye la diagonal de azar, leyenda legible y el AUC de cada modelo. Si es viable, unifica las cinco curvas en un mismo gráfico para comparar.
- Explica en pocas líneas que ROC-AUC evalúa discriminación a varios umbrales y que no reemplaza el análisis de Recall_M ni falsos negativos.

**Resultado esperado / criterio de aceptación**

Figura ROC reproducible de los cinco modelos y explicación coherente de sus AUC.

**Atención:** No usar `y_pred` binario como entrada a `roc_curve` cuando existen scores/probabilidades.

### 5.5. Análisis de errores y elección del modelo

**Responsable: Marco.**

**Qué debes hacer**

- Cruza la tabla de métricas con las matrices de confusión: identifica cuál(es) detectan más casos M y cuál(es) generan menos falsos negativos.
- Propón un modelo preferido **para el ejercicio académico**, justificando el equilibrio entre Recall_M, Precision_M, F1_M y ROC_AUC y aclarando las limitaciones del conjunto de prueba pequeño.
- Explica por qué seleccionar según Accuracy exclusivamente puede ser inapropiado si la prioridad son las muestras malignas.
- No afirmar que el mejor modelo sirve para uso clínico real ni ajustar umbrales sobre el test para reportar el mismo test como validación imparcial.

**Resultado esperado / criterio de aceptación**

Selección argumentada y basada en resultados reales, sin sobreinterpretar el rendimiento.

## 6. Interpretación y discusión

### 6.1. Importancia de características en Random Forest

**Responsable: Vera.**

**Qué debes hacer**

- Recupera `feature_importances_` del modelo `Random Forest` entrenado en 4.5 y vincúlalo con los nombres originales de `X.columns`.
- Muestra una gráfica horizontal con las 10 características de mayor importancia.
- Explica que las importancias internas del bosque indican contribución al modelo, **no causalidad** ni relevancia médica validada.

**Resultado esperado / criterio de aceptación**

Gráfico legible de top 10 de características y explicación breve de sus limitaciones.

### 6.2. Síntesis del EDA frente a los resultados

**Responsable: Aldana.**

**Qué debes hacer**

- Relaciona dos o tres hallazgos del EDA (balance, distribuciones o correlaciones) con lo observado en el rendimiento de los modelos.
- Distingue hechos observados de interpretaciones; evita afirmar que cierta variable causa malignidad.
- Enumera limitaciones metodológicas: tamaño del dataset, variabilidad de una partición train/test y ausencia de validación externa.

**Resultado esperado / criterio de aceptación**

Párrafo o lista breve de hallazgos y limitaciones respaldados por las salidas del notebook.

### 6.3. Conclusiones, reproducibilidad y recomendaciones

**Responsable: Marco.**

**Qué debes hacer**

- Resume el objetivo, el método de separación, el escalamiento aplicado solo al entrenamiento y los cinco modelos comparados.
- Indica el mejor resultado **con valores numéricos que aparezcan realmente en `resultados_df`**, así como al menos una limitación.
- Propón como trabajo futuro validación cruzada, ajuste de hiperparámetros y evaluación externa; deja claro que son recomendaciones y no trabajos ya realizados.
- Verifica que todas las gráficas tienen títulos, etiquetas y que ninguna conclusión contradiga los valores observados.

**Resultado esperado / criterio de aceptación**

Conclusiones completas, no genéricas, coherentes con las evidencias del notebook.

## 7. Control final de integración

### 7.1. Verificación y entrega en Colab

**Responsable: Marco.**

**Qué debes hacer**

- Abre la versión final en Google Colab, usa **Reiniciar sesión y ejecutar todo** y comprueba que todas las celdas terminan sin excepción.
- Comprueba que en un entorno limpio se puede obtener `df_raw`, `X`, `y`, la división train/test, los cinco modelos, `resultados_df` y todas las figuras.
- Confirma que los resultados se han calculado con etiquetas 1=M y 0=B y que no hubo `fit` del escalador sobre test ni preprocesamiento con información del test.
- Revisa los nombres de responsables en celdas de texto, la ortografía, títulos, leyendas y que no haya salidas de errores ni celdas experimentales sobrantes.
- Registra de forma breve la fuente del dataset, las versiones principales de librerías si resulta útil, y el nombre del archivo final compartido con el grupo.

**Resultado esperado / criterio de aceptación**

Notebook listo para entregar: se ejecuta de arriba abajo en Colab y conserva evidencias de todas las etapas.

---

### Lista de comprobación de entrega

- [ ] **Marco:** configuración, contrato de variables, cálculo centralizado de métricas, análisis de errores y cierre.
- [ ] **Poma:** carga verificable, limpieza, `X`/`y` correctamente definidos y tabla de resultados.
- [ ] **Aldana:** tres bloques de EDA, regresión logística, curvas ROC y síntesis.
- [ ] **Brickzon:** partición estratificada, escalamiento sin fuga, controles de preparación, KNN y SVM.
- [ ] **Vera:** árbol de decisión, Random Forest, matrices de confusión e importancia de características.
- [ ] **Equipo:** todos los modelos están registrados; se respetó `1 = M`; se compararon los cinco clasificadores sobre el **mismo** conjunto de prueba.
- [ ] **Marco (integración):** ejecución completa en Google Colab desde sesión limpia, sin errores y con conclusiones basadas en métricas reales.

> **Estado inicial de esta plantilla:** todas las celdas de código están deliberadamente vacías. Marcar casillas solamente después de implementar y verificar cada entrega.